In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from sklearn.cluster import *
from mlxtend.frequent_patterns import apriori, association_rules
from sklearn.preprocessing import StandardScaler

In [ ]:
data = pd.read_csv('../dataset_clean_2.csv', delimiter=',', dtype={
    'PRIORITY': 'category',
    'COMPONENT': 'category',
    'LABEL': 'category',
    'URGENCY': 'category',
    'IMPACT': 'category',
    'ISSUE_CATEGORY': 'category',
    'ASSIGNEE': 'category',
    'POZISYON': 'category',
    'IS_BILGISI': 'category',
    'Kademe': 'category',
    'Complition Time': 'float32',
    'Total_Assignee': 'int8',
    'Total_Worklog_Assginee': 'int8',
    'Total_Log_Hours_Assignee': 'float32',
    'COMMENTOR_COUNT': 'int8',
    'COMMENT_COUNT': 'int8',
    'WAITING_HOURS_AT_CUSTOMER': 'float32',
    'ISSUE_TYPE': 'category',
})

numerical_features = [
    'Complition Time',
    'Total_Assignee',
    'Total_Worklog_Assginee',
    'Total_Log_Hours_Assignee',
    'COMMENTOR_COUNT',
    'COMMENT_COUNT',
    'WAITING_HOURS_AT_CUSTOMER'
]

In [ ]:
data.head()

## Generate combinations of numerical features

In [ ]:
feature_comb = list(combinations(numerical_features, 2))

In [ ]:
def make_cluster(algorithm, data, features, figsize, **kwargs):
    fig, (ax1, ax2) = plt.subplots(2, figsize=figsize)
    fig.suptitle('Ticket Clustering')
    sns.scatterplot(ax=ax1, data=data, x=features[0], y=features[1], hue='URGENCY', palette='viridis')
    ax1.set(title='Ground Truth', xlabel=features[0], ylabel=features[1])
    algorithm_instance = algorithm(**kwargs)
    algorithm_name = type(algorithm_instance).__name__
    data['Cluster'] = algorithm_instance.fit_predict(data[features])
    sns.scatterplot(ax=ax2, data=data, x=features[0], y=features[1], hue='Cluster', palette='viridis')
    ax2.set(title=f"Clustering by {algorithm_name}", xlabel=features[0], ylabel=features[1])
    filename = f"{algorithm_name}_{'_'.join(features)}.png"
    # Save the figure with the generated filename
    plt.savefig("./figures/" + filename)

## DBSCAN

In [ ]:
for fc in feature_comb:
    features = [fc[0], fc[1]]
    make_cluster(DBSCAN, data, features, (12, 10), eps=3, min_samples=5)

## Kmeans

In [ ]:
for fc in feature_comb:
    features = [fc[0], fc[1]]
    make_cluster(KMeans, data, features, (12, 10), n_clusters=3, random_state=42)

## Agglomerative Clustering

In [ ]:
for fc in feature_comb:
    features = [fc[0], fc[1]]
    scaler = StandardScaler()
    data_scaled = scaler.fit_transform(data[features])
    make_cluster(AgglomerativeClustering, data, features, (12, 10), n_clusters=3, metric='euclidean', linkage='ward')

## Apriori

In [ ]:
data = pd.get_dummies(data)
columns = [
    'IMPACT_Operation Continue With Workaround',
    'PRIORITY_Major',
    'IMPACT_Some Processes Stop on Production',
    'IMPACT_Operation Continue With Workaround',
    'PRIORITY_Minor'
]

# Apply Apriori algorithm
frequent_itemsets = apriori(data[columns], min_support=0.05, use_colnames=True)

# Generate association rules
association_rules_df = association_rules(frequent_itemsets, metric="lift", min_threshold=1)

# Sort rules by lift
association_rules_df = association_rules_df.sort_values(by='lift', ascending=False)

# Print the top 10 interesting patterns
top_patterns = association_rules_df.head(10)
print(top_patterns)